In [ ]:
import matplotlib
matplotlib.use("Agg")
import os
import gc
import getpass
import copy
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms

from torchmetrics.classification import MulticlassAccuracy
from tqdm.auto import tqdm


# Device Agnostic Code

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

user_account = getpass.getuser()

# Path Configuratons 

script_dir = os.getcwd()
train_dir = os.path.join(script_dir, "Food Vision Data", "train")
test_dir = os.path.join(script_dir, "Food Vision Data", "test")


# Torchvision Preprocessing Pipelines

train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),            
    transforms.RandomHorizontalFlip(p=0.5),   
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ToTensor(),                    
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225]) 
])

test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Image Folder Datasets & Batch Dataloaders

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

BATCH_SIZE = 50
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, pin_memory=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, pin_memory=True, num_workers=4)

# Class Counts

class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Dataset Classes Found: {[a for a in class_names]} ({num_classes} total)\n")

# Production 2D CNN Architecture

class ProductionCNN(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = num_classes, dropout_rate: float = 0.4):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32), 
            nn.LeakyReLU(negative_slope=0.1),
            nn.MaxPool2d(kernel_size=2, stride=2), # Compresses height and width by half

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(negative_slope=0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=5, stride=2, padding=3),
            nn.BatchNorm2d(129),
            nn.LeakyReLU(negative_slope=0.3),
            nn.MaxPool2d(kernel_size=3, stride=1),            
            
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(negative_slope=0.1),
            
            nn.AdaptiveAvgPool2d((1, 1)) # Yields shape [ Batch, 128, 1, 1 ]
        )
        
        # Classification Network 

        self.classifier = nn.Sequential(
            nn.Flatten(), 
            
            nn.Linear(in_features=256, out_features=64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Dropout(p=dropout_rate), 
            
            nn.Linear(in_features=64, out_features=num_classes) 
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x

# Grid Search 

def grid_search(learning_rates : list, dropout_rates:list, epochs : float):

    all_final_accuracies = []
    all_configs = []
    all_model_states = []

    print("--- Initializing Grid Search Optimization Loop ---")

    # Grid Search Execution Loops

    for lr in learning_rates:
        for dropout in dropout_rates:
            print(f"\nEvaluating Configuration : LR = {lr} | Dropout = {dropout}")
            
            model = ProductionCNN(dropout_rate=dropout).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            
            accuracy_metric = MulticlassAccuracy(num_classes=num_classes).to(device)

            # Training & Testing Loop

            TUNING_EPOCHS = epochs
            for epoch in tqdm(range(TUNING_EPOCHS)):
                model.train()
                for images, labels in train_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                
                # Evaluation Pass

                model.eval()
                accuracy_metric.reset()

                with torch.inference_mode():
                    for images, labels in test_loader:
                        images, labels = images.to(device), labels.to(device)
                        outputs = model(images)
                        preds = torch.argmax(outputs, dim=1)
                        accuracy_metric.update(preds, labels)
                
                current_epoch_acc = accuracy_metric.compute().item()
                
            print(f"-> Final Epoch Accuracy: {current_epoch_acc:.4f}")
            
            # Append straight into the flat tracking lists

            all_final_accuracies.append(current_epoch_acc)
            all_configs.append(f"LR={lr}, Dropout={dropout}")
            all_model_states.append(copy.deepcopy(model.state_dict()))
                
            # Free Memory
            del model, optimizer, criterion, accuracy_metric
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return all_final_accuracies, all_configs, all_model_states

param_grid = {
    'lr': [0.001, 0.0001],
    'dropout': [0.3, 0.5]
}
all_final_accuracies, all_configs, all_model_states = grid_search(param_grid["lr"], param_grid["dropout"], 560)



print(f"\n--- Grid Search Optimization Complete ---")

# Extracting the Absolute Best and Worst with One-Liners

best_acc = max(all_final_accuracies)
worst_acc = min(all_final_accuracies)

best_index = all_final_accuracies.index(best_acc)
worst_index = all_final_accuracies.index(worst_acc)

print(f"🥇 BEST CONFIG: {all_configs[best_index]} | Max Accuracy: {best_acc:.4f}")
print(f"❌ WORST CONFIG: {all_configs[worst_index]} | Min Accuracy: {worst_acc:.4f}")

# Create a Fresh Model to load Fresh Weights

print("--- Loading Fresh Champion Instance for Predictions ---")
fresh_model = ProductionCNN(in_channels=3, num_classes=num_classes).to(device)
fresh_model.load_state_dict(all_model_states[best_index])
fresh_model.eval()

inference_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Ready for live user inference.\n")

while True:
    user_image_path = input("Enter the absolute path to an image file ( or type 'exit' to quit ): ").strip()
    
    if user_image_path.lower() == 'exit':
        print("Exiting prediction loop.")
        break
        
    if not os.path.exists(user_image_path):
        print("❌ Error: File path does not exist. Please try again.")
        continue
        
    try:
        raw_img = Image.open(user_image_path).convert("RGB")
        input_tensor = inference_transforms(raw_img).unsqueeze(0).to(device)
        
        with torch.inference_mode():
            logits = fresh_model(input_tensor)
            pred_probs = torch.softmax(logits, dim=1)
            pred_idx = torch.argmax(pred_probs, dim=1).item()
            confidence = pred_probs[pred_idx].item() * 100
            
        print(f"\n== TARGET PREDICTION RESULTS ==")
        print(f"Predicted Class: {class_names[pred_idx].upper()}")
        print(f"Confidence Score: {confidence:.2f}%")
        print("-" * 40 + "\n")
        
    except BaseException as e:
        print(f"❌ Failed to process image file. Error: {e}")

try:
    plt.figure(figsize=(14,5))
    plt.imshow(input_tensor)
    plt.title(f"Prediction = {class_names[pred_idx].title()} | Confidence = {confidence} %", c="green")
except BaseException as e:
    print(f"Failed to Graph your Image")
